# 02 Provider Routing and Model Policy (OpenClaw, 2026)

## What This Lesson Is
Design explicit model/provider routing policy and apply it through OpenClaw configuration commands.

## Scientific Lens
- Concept: Policy-driven provider selection by sensitivity and latency budget.
- Measure: Policy compliance rate across deterministic request set.
- Validity Limit: Routing quality depends on accurate request metadata classification.


## How It Works
1. Define deterministic routing table from workload class to provider/model.
2. Validate policy outputs for representative requests.
3. Apply primary model config live and verify model listing.


In [ ]:
import os
import shutil

HAS_OPENCLAW = shutil.which("openclaw") is not None
print("openclaw available:", HAS_OPENCLAW)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL", "<unset>"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: runs real OpenClaw CLI operations when available; otherwise prints explicit skip guidance.


In [ ]:
# Deterministic Demo
requests = [
    {"id": "r1", "class": "pii", "latency_ms": 1200},
    {"id": "r2", "class": "general", "latency_ms": 500},
    {"id": "r3", "class": "code", "latency_ms": 700},
]
policy = {"pii": "openai/gpt-5.1-codex", "general": "ollama/qwen2.5-coder:1.5b", "code": "openai/gpt-5.1-codex"}
routes = {r["id"]: policy[r["class"]] for r in requests}
print(routes)
assert routes["r1"].startswith("openai/")
assert routes["r2"].startswith("ollama/")


In [ ]:
# Live Demo
import os
import shutil
import subprocess

if shutil.which("openclaw") is None:
    print("Skipping live policy demo: openclaw CLI is not installed.")
else:
    model = os.getenv("OPENCLAW_OPENAI_MODEL", "openai/gpt-5.1-codex")
    for cmd in (["openclaw", "config", "set", "agents.defaults.model.primary", model], ["openclaw", "models", "list"]):
        print("$", " ".join(cmd))
        proc = subprocess.run(cmd, capture_output=True, text=True)
        print((proc.stdout or proc.stderr).strip()[:1200])


## Applied Labs
1. Add a `cost_cap_cents` field and route to local model when cap is low.
2. Add fallback policy for unknown workload classes and test behavior.
3. Emit decision logs with request id and selected model.

## Validation Checklist
- Each workload class maps to exactly one explicit model id.
- Deterministic policy output is asserted for at least 3 classes.
- Live config command result is captured for auditability.

## Further Reading
- OpenClaw docs: https://docs.openclaw.ai
- NIST AI RMF: https://www.nist.gov/itl/ai-risk-management-framework
- Routing patterns: https://martinfowler.com/articles/patterns-of-distributed-systems/
